In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# ========== Load Files ==========
df1 = pd.read_csv("Normal_data.csv")
df2 = pd.read_csv("metasploitable-2.csv")
df3 = pd.read_csv("OVS.csv")

# ========== Merge ==========
df = pd.concat([df1, df2, df3], axis=0).reset_index(drop=True)

# ========== Clean ==========
df = df.drop_duplicates()
df = df.dropna()

# ========== Encode Categorical ==========
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

# ========== Identify Label Column ==========
# Change if your label column name is different
label_col = "Label"

X = df.drop(label_col, axis=1)
y = df[label_col]

# ========== Normalize ==========
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# ========== Feature Extraction: PCA to 15 ==========
pca = PCA(n_components=15)
X_pca = pca.fit_transform(X_scaled)

# ========== Train Test Split ==========
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print("Final Shapes:")
print("Train:", X_train.shape, "Test:", X_test.shape)


Final Shapes:
Train: (275110, 15) Test: (68778, 15)


In [2]:
from tensorflow.keras.layers import Input, GRU, Dense
from tensorflow.keras.models import Model

inp = Input(shape=(1, X_train.shape[1]))
x = GRU(32)(inp)
out = Dense(X_train.shape[1], activation='linear')(x)

gru_model = Model(inp, out)
gru_model.compile(optimizer='adam', loss='mse')


In [3]:
X_train_r = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_r = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])


In [ ]:
gru_model.fit(X_train_r, X_train, epochs=20, batch_size=64)


Epoch 1/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 31s 5ms/step - loss: 0.0112
Epoch 2/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - loss: 7.3791e-06
Epoch 3/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - loss: 3.5708e-06
Epoch 4/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - loss: 2.5029e-06
Epoch 5/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - loss: 1.9833e-06
Epoch 6/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - loss: 1.7356e-06
Epoch 7/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - loss: 1.5494e-06
Epoch 8/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - loss: 1.3057e-06
Epoch 9/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - loss: 1.4232e-06
Epoch 10/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 24s 6ms/step - loss: 1.3304e-06
Epoch 11/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - loss: 1.3181e-06
Epoch 12/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 24s 5ms/step - loss: 1.3110e-06
Epoch 13/20
4299/4299 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - loss: 1.0958e-06
Epoch 14/20
4299/4299 ━━━━━━━━━━━━━━━━

In [5]:
gru_encoder = Model(inputs=inp, outputs=x)


In [6]:
X_train_gru = gru_encoder.predict(X_train_r)
X_test_gru = gru_encoder.predict(X_test_r)

print("GRU Feature Shape:", X_train_gru.shape)   # (samples, 32)


8598/8598 ━━━━━━━━━━━━━━━━━━━━ 29s 3ms/step
2150/2150 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step
GRU Feature Shape: (275110, 32)


In [7]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

gb = GradientBoostingClassifier()
gb.fit(X_train_gru, y_train)


GradientBoostingClassifier()

In [9]:
y_pred2 = gb.predict(X_test_gru)

print("Hybrid Model 2: GRU + Gradient Boosting")
print("Accuracy:", accuracy_score(y_test, y_pred2))
print(classification_report(y_test, y_pred2))


Hybrid Model 2: GRU + Gradient Boosting
Accuracy: 0.9966122888132833
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       281
           1       0.94      0.91      0.92        33
           2       1.00      0.99      1.00     14706
           3       1.00      1.00      1.00      9683
           4       0.99      0.99      0.99     10723
           5       1.00      1.00      1.00     13685
           6       1.00      1.00      1.00     19626
           7       0.00      0.00      0.00         3
           8       0.51      0.97      0.67        38

    accuracy                           1.00     68778
   macro avg       0.83      0.87      0.84     68778
weighted avg       1.00      1.00      1.00     68778

